In [5]:
import pandas as pd
# df_final_f = pd.read_csv('mor_data/final_df_with_all_scores_clusters_llama_clean.csv')
df_final_f = pd.read_csv('icl_data_to_share_fk/llama/54_73/seed_4/k_mean_buckets/bucket_assignments/all_settings_bucket_assignments_k3.csv')

In [12]:
df_final_f

,case_id,setting,subject,subject_id,target_new,target_true,avg_new_dominance_generation,avg_true_dominance_generation,avg_dominance_diff_generation,relation_id,popularity_score,avg_conflict_pos,avg_true_dominance_generation_bucket_k3,avg_new_dominance_generation_bucket_k3,avg_dominance_diff_generation_bucket_k3,avg_conflict_pos_bucket_k3
0,21,encyclopaedia,Argentine Football Association,Q496548,NATO,FIFA,0.2981,0.3288,0.0308,P463,203,0.000000,2,2,2,1
1,56,encyclopaedia,Toyota Cresta,Q1815602,BMW,Toyota,0.1390,0.6476,0.5085,P176,194,0.003519,3,1,3,1
2,304,encyclopaedia,Jari Kurri,Q363621,soccer,hockey,0.2684,0.7184,0.4500,P641,101,0.044955,3,2,3,2
3,366,encyclopaedia,New Bedford Whaling Museum,Q7005471,Dublin,Massachusetts,0.2854,0.4458,0.1604,P131,19,0.000000,2,2,2,1
4,378,encyclopaedia,The Paradise Club,Q3895015,NBC,BBC,0.6937,0.0688,-0.6250,P449,17,0.002424,1,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,21440,neutral_new,Lu Watters,Q1872759,trance,jazz,0.1274,0.5403,0.4129,P136,7,0.000010,3,1,3,1
896,21681,neutral_new,Frank Pembleton,Q5488954,Paris,Baltimore,0.3529,0.5500,0.1971,P937,134,0.001620,3,2,2,1
897,21728,neutral_new,Gustave Geffroy,Q927796,Italian,French,0.1794,0.6471,0.4676,P103,6,0.002996,3,1,3,1
898,21759,neutral_new,Oleg Kotov,Q342338,English,Russian,0.2375,0.5150,0.2775,P103,5,0.011649,3,1,3,1


In [1]:
import torch
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

def extract_layer_embeddings(
    sentences: list[str], 
    model: AutoModel, 
    tokenizer: AutoTokenizer, 
    layer_index: int, 
    batch_size: int = 8,
    pooling: str = 'mean',  # 'mean' or 'cls'
    device: str = 'cpu'
) -> list[torch.Tensor]:
    """
    Extracts token embeddings from a specific layer of an autoregressive Hugging Face model.
    
    Args:
        sentences: List of input sentences.
        model: Hugging Face model (e.g., GPT-2, Llama).
        tokenizer: Hugging Face tokenizer corresponding to the model.
        layer_index: The index of the layer to extract. 
                     0 is the initial embeddings, 1 is the first transformer block, 
                     -1 is the final layer.
        batch_size: Number of sentences to process at once.
        device: Device to run the model on ('cpu', 'cuda', etc.).
        
    Returns:
        A list of 2D tensors of shape (sequence_length, hidden_size), 
        one for each input sentence (padding removed).
    """
    # model.to(device)
    model.eval()

    # Autoregressive models often don't have a default pad token.
    # We set it to the EOS token to allow for batched processing.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    all_embeddings = []

    for i in tqdm(range(0, len(sentences), batch_size), desc="Processing batches"):
        batch_sentences = sentences[i:i + batch_size]

        # Tokenize the batch
        inputs = tokenizer(
            batch_sentences, 
            padding=True, 
            truncation=True, 
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            # Crucial: output_hidden_states=True tells the model to return all layer states
            outputs = model(**inputs, output_hidden_states=True)

        # outputs.hidden_states is a tuple of length (num_layers + 1)
        # It includes the initial embedding layer followed by all transformer blocks.
        hidden_states = outputs.hidden_states
        
        # Select the requested layer. Shape: (batch_size, seq_len, hidden_size)
        layer_states = hidden_states[layer_index]

        # Remove padding and add to our final list
        attention_mask = inputs['attention_mask']
        for j in range(len(batch_sentences)):
            # Calculate actual sequence length ignoring padding
            seq_len = attention_mask[j].sum().item()
            
            # Slice the tensor to keep only the actual tokens (discard padding)
            # For right-padded sequences, valid tokens are at the beginning.
            if pooling == 'mean':
                actual_embeddings = layer_states[j, :seq_len, :].cpu().mean(dim=0)  # Shape: (hidden_size,)
                all_embeddings.append(actual_embeddings)
            else:
                raise NotImplementedError("Only 'mean' pooling is implemented in this function.")

    return all_embeddings

/home/ss20428/miniconda3/envs/multi_312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
torch.cuda.device_count()

1

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.3-70B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 30/30 [03:19<00:00,  6.64s/it]


In [3]:
def extract_layer0_embeddings(
    sentences: list[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    batch_size: int = 64,
    device: str = "cuda",
) -> torch.Tensor:
    """
    Mean-pool the token embedding table (layer 0) for each sentence.
    Directly calls embed_tokens — no transformer layers run, so it is fast.
    Returns a (N, hidden_dim) float32 tensor on CPU.
    """
    embed = model.model.embed_tokens  # works for Llama / Mistral family
    # embed = embed.to(device)
    embed.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), batch_size), desc="Embedding"):
            batch = sentences[i : i + batch_size]
            inputs = tokenizer(
                batch, padding=True, truncation=True, return_tensors="pt"
            ).to(embed.weight.device)
            print(inputs["input_ids"].device)
            # print(embed.weight.device)
            hidden = embed(inputs["input_ids"]).float()   # (B, T, D)
            mask = inputs["attention_mask"]               # (B, T)
            # mean-pool over non-padding tokens
            lengths = mask.sum(dim=1, keepdim=True).float()   # (B, 1)
            pooled = (hidden * mask.unsqueeze(-1)).sum(dim=1) / lengths  # (B, D)
            all_embs.append(pooled.cpu())
    return torch.cat(all_embs, dim=0)


In [6]:
with torch.no_grad():
    model.eval()
    prompt_embeds = extract_layer0_embeddings(
        sentences=((df_final_f['subject'])).tolist(),
        model=model,
        tokenizer=tokenizer,
        device='cpu'
    )

    target_true_embeds = extract_layer0_embeddings(
        sentences=((df_final_f['target_true'])).tolist(),
        model=model,
        tokenizer=tokenizer,
        device='cpu'
    )

    target_new_embeds = extract_layer0_embeddings(
        sentences=((df_final_f['target_new'])).tolist(),
        model=model,
        tokenizer=tokenizer,
        device='cpu'
    )

    sentence_embeddings = torch.cat([target_true_embeds, target_new_embeds], dim=1)
    sentence_embeddingssubject_counter = torch.cat([prompt_embeds, target_new_embeds], dim=1)




Embedding:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding: 100%|██████████| 15/15 [00:00<00:00, 77.94it/s]


cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu


Embedding:   0%|          | 0/15 [00:00<?, ?it/s]

cpu
cpu
cpu
cpu
cpu


Embedding: 100%|██████████| 15/15 [00:00<00:00, 363.30it/s]


cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu


Embedding: 100%|██████████| 15/15 [00:00<00:00, 438.03it/s]

cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu


In [ ]:
# with torch.no_grad():
#     model.eval()
#     target_new_embeddings = extract_layer0_embeddings(
#         sentences=df_final_f['target_new'].tolist(),
#         model=model,
#         tokenizer=tokenizer,
#         # layer_index=0,  
#         # pooling='mean',
#         device='cpu'
#     )

#     target_true_embeddings = extract_layer0_embeddings(
#         sentences=df_final_f['target_true'].tolist(),
#         model=model,
#         tokenizer=tokenizer,
#         # layer_index=0,
#         # pooling='mean',
#         device='cpu'
#     )


Embedding: 100%|██████████| 8/8 [00:00<00:00, 454.68it/s]


In [ ]:
# def train_classifier(df_):
#     from sklearn.linear_model import LogisticRegression
#     from sklearn.model_selection import train_test_split

#     X = df_[['target_true_entity_popularity', 'target_new_entity_popularity']]
#     y = df_['bddiff_cluster']
#     print("The distribution of clusters:", y.value_counts())

#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42)

#     clf = LogisticRegression()
#     clf.fit(X_train, y_train)

#     print("Train Accuracy:", clf.score(X_train, y_train))
#     print("Accuracy:", clf.score(X_test, y_test))

#     # classification reports
#     from sklearn.metrics import classification_report
#     y_pred_train = clf.predict(X_train)
#     y_pred_test = clf.predict(X_test)
#     print("Classification Report (Train):")
#     print(classification_report(y_train, y_pred_train))
#     print("Classification Report (Test):")
#     print(classification_report(y_test, y_pred_test))

#     print("Coefficients:", clf.coef_, clf.intercept_)
#     return y.mean(), clf.score(X_test, y_test), clf.score(X_train, y_train)

In [ ]:
# results = []
# manipulations = df_final['setting_x'].unique()
# for man in manipulations:
#     print(f"Training classifier for manipulation type: {man}")
#     get_indices = df_final['setting_x'] == man
#     get_indices = get_indices.values
#     print(get_indices)
#     results.append((man, ) + train_classifier(df_final[get_indices]))
#     print()

In [ ]:
# train_classifier(df_final)

In [ ]:
# import numpy as np
# def train_classifier_with_embeddings(df_, target_true_embeddings, target_new_embeddings, target_y='bd_basecluster'):
#     from sklearn.linear_model import LogisticRegression

#     # get sklearn mlp
#     from sklearn.neural_network import MLPClassifier    

#     from sklearn.model_selection import train_test_split
#     # get functions for classification report 
#     from sklearn.metrics import classification_report, confusion_matrix

#     # Convert list of tensors to a single 2D array
#     # target_true_embeddings_array = torch.stack(target_true_embeddings).cpu().numpy()
#     # target_new_embeddings_array = torch.stack(target_new_embeddings).cpu().numpy()

#     # Combine the two sets of embeddings into one feature set
#     X = np.concatenate([target_true_embeddings.cpu().numpy(), target_new_embeddings.cpu().numpy()], axis=1)
#     y = df_[target_y].values
#     print("How many change:", y.sum(), "out of", len(y), "ratio: ", y.mean())

#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=27)

#     clf = LogisticRegression(max_iter=2000)
#     clf.fit(X_train, y_train)

#     print("Train Accuracy:", clf.score(X_train, y_train))
#     print("Test Accuracy:", clf.score(X_test, y_test))
#     # print("Coefficients:", clf.coef_, clf.intercept_)
#     get_f1 = classification_report(y_test, clf.predict(X_test))
#     print("Classification Report:\n", get_f1)
#     get_f1_train = classification_report(y_train, clf.predict(X_train))
#     print("Classification Report (Train):\n", get_f1_train)
#     return y.mean(), clf.score(X_test, y_test), clf.score(X_train, y_train)

In [7]:
import numpy as np


def train_classifier_with_embeddings_balanced_train(
    df_,
    sentence_embeddings,
    target_y="bd_basecluster",
    balance="undersample",
    test_split=0.5,
    split_random_state=27,
    print_mode=True,
):
    from sklearn.linear_model import LogisticRegression
    from sklearn.neural_network import MLPClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix

    # Combine the two sets of embeddings into one feature set
    X = sentence_embeddings.cpu().numpy()
    y = df_[target_y].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_split, random_state=split_random_state
    )

    # Balance the training set
    rng = np.random.default_rng(split_random_state)
    classes, counts = np.unique(y_train, return_counts=True)
    if balance == "undersample":
        target_count = counts.min()
        keep_idx = np.concatenate(
            [
                rng.choice(np.where(y_train == c)[0], size=target_count, replace=False)
                for c in classes
            ]
        )
        if print_mode:
            print(
                f"Undersampled train set: {target_count} samples per class ({len(classes)} classes)"
            )
    elif balance == "oversample":
        target_count = counts.max()
        keep_idx = np.concatenate(
            [
                rng.choice(np.where(y_train == c)[0], size=target_count, replace=True)
                for c in classes
            ]
        )
        if print_mode:
            print(
                f"Oversampled train set: {target_count} samples per class ({len(classes)} classes)"
            )
    else:
        raise ValueError(
            f"balance must be 'undersample' or 'oversample', got '{balance}'"
        )

    X_train, y_train = X_train[keep_idx], y_train[keep_idx]

    clf = LogisticRegression(max_iter=2000, l1_ratio=0, C=0.5)
    clf.fit(X_train, y_train)

    if print_mode:
        print("Train Accuracy:", clf.score(X_train, y_train))
        print("Test Accuracy:", clf.score(X_test, y_test))

    get_f1 = classification_report(y_test, clf.predict(X_test))

    # make a balanced test set for evaluation
    rng = np.random.default_rng(27)
    classes, counts = np.unique(y_test, return_counts=True)
    target_count = min(counts)
    keep_idx = np.concatenate(
        [
            rng.choice(np.where(y_test == c)[0], size=target_count, replace=False)
            for c in classes
        ]
    )
    X_test_balanced, y_test_balanced = X_test[keep_idx], y_test[keep_idx]
    get_f1_balanced = classification_report(
        y_test_balanced, clf.predict(X_test_balanced)
    )

    if print_mode:
        print("Classification Report (Balanced Test Set):\n", get_f1_balanced)
    if print_mode:
        print("Classification Report:\n", get_f1)

    get_f1_train = classification_report(y_train, clf.predict(X_train))
    if print_mode:
        print("Classification Report (Train):\n", get_f1_train)
    return {
        "test_accuracy": clf.score(X_test, y_test),
        "balanced_test_accuracy": clf.score(X_test_balanced, y_test_balanced),
        "train_accuracy": clf.score(X_train, y_train),
        "classification_report_test": get_f1,
        "classification_report_train": get_f1_train,
        "classification_report_balanced_test": get_f1_balanced,
    }

In [13]:
len(df_final_f)

450

In [42]:
# target_true_embeddings = torch.stack(target_true_embeddings)
# target_new_embeddings = torch.stack(target_new_embeddings)

In [43]:
# # save the embeddings as numpy arrays
# np.save('target_true_embeddings_for_relabled.npy', target_true_embeddings.cpu().numpy())
# np.save('target_new_embeddings_for_relabled.npy', target_new_embeddings.cpu().numpy())

In [ ]:
# results = []
# manipulations = df_final_f['setting_x'].unique()
# for man in manipulations:
#     print(f"Training classifier for manipulation type: {man}")
#     get_indices = df_final_f['setting_x'] == man
#     get_indices = get_indices.values
#     print(get_indices)
#     results.append((man, ) + train_classifier_with_embeddings(df_final_f[get_indices], target_true_embeddings[get_indices], target_new_embeddings[get_indices]))
#     print()

In [10]:
train_classifier_with_embeddings_balanced_train(df_final_f, sentence_embeddingssubject_counter, target_y='avg_true_dominance_generation_bucket_k3', balance='oversample')

Oversampled train set: 184 samples per class (3 classes)
Train Accuracy: 0.8278985507246377
Test Accuracy: 0.4777777777777778
Classification Report (Balanced Test Set):
               precision    recall  f1-score   support

           1       0.59      0.55      0.57        85
           2       0.40      0.45      0.42        85
           3       0.53      0.49      0.51        85

    accuracy                           0.50       255
   macro avg       0.50      0.50      0.50       255
weighted avg       0.50      0.50      0.50       255

Classification Report:
               precision    recall  f1-score   support

           1       0.36      0.55      0.44        85
           2       0.51      0.44      0.47       202
           3       0.54      0.48      0.51       163

    accuracy                           0.48       450
   macro avg       0.47      0.49      0.47       450
weighted avg       0.49      0.48      0.48       450

Classification Report (Train):
             

{'test_accuracy': 0.4777777777777778,
 'balanced_test_accuracy': 0.4980392156862745,
 'train_accuracy': 0.8278985507246377,
 'classification_report_test': '              precision    recall  f1-score   support\n\n           1       0.36      0.55      0.44        85\n           2       0.51      0.44      0.47       202\n           3       0.54      0.48      0.51       163\n\n    accuracy                           0.48       450\n   macro avg       0.47      0.49      0.47       450\nweighted avg       0.49      0.48      0.48       450\n',
 'classification_report_train': '              precision    recall  f1-score   support\n\n           1       0.82      0.92      0.86       184\n           2       0.88      0.73      0.80       184\n           3       0.80      0.84      0.82       184\n\n    accuracy                           0.83       552\n   macro avg       0.83      0.83      0.83       552\nweighted avg       0.83      0.83      0.83       552\n',
 'classification_report_bal

In [11]:
train_classifier_with_embeddings_balanced_train(df_final_f, sentence_embeddingssubject_counter, target_y='avg_new_dominance_generation_bucket_k3', balance='oversample')


Oversampled train set: 266 samples per class (3 classes)
Train Accuracy: 0.8471177944862155
Test Accuracy: 0.5444444444444444
Classification Report (Balanced Test Set):
               precision    recall  f1-score   support

           1       0.52      0.61      0.56        61
           2       0.42      0.43      0.42        61
           3       0.62      0.51      0.56        61

    accuracy                           0.51       183
   macro avg       0.52      0.51      0.51       183
weighted avg       0.52      0.51      0.51       183

Classification Report:
               precision    recall  f1-score   support

           1       0.71      0.59      0.65       249
           2       0.44      0.47      0.46       140
           3       0.34      0.51      0.41        61

    accuracy                           0.54       450
   macro avg       0.50      0.52      0.50       450
weighted avg       0.58      0.54      0.55       450

Classification Report (Train):
             

{'test_accuracy': 0.5444444444444444,
 'balanced_test_accuracy': 0.5136612021857924,
 'train_accuracy': 0.8471177944862155,
 'classification_report_test': '              precision    recall  f1-score   support\n\n           1       0.71      0.59      0.65       249\n           2       0.44      0.47      0.46       140\n           3       0.34      0.51      0.41        61\n\n    accuracy                           0.54       450\n   macro avg       0.50      0.52      0.50       450\nweighted avg       0.58      0.54      0.55       450\n',
 'classification_report_train': '              precision    recall  f1-score   support\n\n           1       0.86      0.82      0.84       266\n           2       0.82      0.82      0.82       266\n           3       0.86      0.90      0.88       266\n\n    accuracy                           0.85       798\n   macro avg       0.85      0.85      0.85       798\nweighted avg       0.85      0.85      0.85       798\n',
 'classification_report_bal

# Cross Validation

In [24]:
seed_list = [4, 5, 21, 42, 67, 12, 19, 16, 27, 31, 37, 45, 52, 88, 99]

## For BD Base

In [33]:
bd_base_test_accuracies = []
bd_base_balanced_test_accuracies = []

for seed in seed_list:
    print(f"Training with random seed {seed}...")
    results = train_classifier_with_embeddings_balanced_train(
        df_final_f,
        sentence_embeddings,
        target_y="bd_basecluster",
        balance="oversample",
        split_random_state=seed,
        print_mode=False,
    )
    bd_base_test_accuracies.append(results["test_accuracy"])
    bd_base_balanced_test_accuracies.append(results["balanced_test_accuracy"])

print("BD Base Cluster - Test Accuracies:", bd_base_test_accuracies)
print("BD Base Cluster - Balanced Test Accuracies:", bd_base_balanced_test_accuracies)
print(
    "BD Base Cluster - Mean and Std Dev Test Accuracy:",
    np.mean(bd_base_test_accuracies),
    np.std(bd_base_test_accuracies),
)
print(
    "BD Base Cluster - Mean and Std Dev Balanced Test Accuracy:",
    np.mean(bd_base_balanced_test_accuracies),
    np.std(bd_base_balanced_test_accuracies),
)

Training with random seed 4...
Training with random seed 5...
Training with random seed 21...
Training with random seed 42...
Training with random seed 67...
Training with random seed 12...
Training with random seed 19...
Training with random seed 16...
Training with random seed 27...
Training with random seed 31...
Training with random seed 37...
Training with random seed 45...
Training with random seed 52...
Training with random seed 88...
Training with random seed 99...
BD Base Cluster - Test Accuracies: [0.5066666666666667, 0.47555555555555556, 0.4311111111111111, 0.43555555555555553, 0.4266666666666667, 0.5111111111111111, 0.4177777777777778, 0.4, 0.4311111111111111, 0.5022222222222222, 0.48444444444444446, 0.5022222222222222, 0.5066666666666667, 0.5333333333333333, 0.4533333333333333]
BD Base Cluster - Balanced Test Accuracies: [0.5161290322580645, 0.5079365079365079, 0.44696969696969696, 0.3508771929824561, 0.40350877192982454, 0.46875, 0.43859649122807015, 0.4146341463414634, 0

## For BD Counter

In [34]:
bd_counter_test_accuracies = []
bd_counter_balanced_test_accuracies = []

for seed in seed_list:
    print(f"Training with random seed {seed}...")
    results = train_classifier_with_embeddings_balanced_train(
        df_final_f,
        sentence_embeddings,
        target_y="bd_counter_cluster",
        balance="oversample",
        split_random_state=seed,
        print_mode=False,
        test_split=0.5,
    )
    bd_counter_test_accuracies.append(results["test_accuracy"])
    bd_counter_balanced_test_accuracies.append(results["balanced_test_accuracy"])

print("BD Counter Cluster - Test Accuracies:", bd_counter_test_accuracies)
print("BD Counter Cluster - Balanced Test Accuracies:", bd_counter_balanced_test_accuracies)
print(
    "BD Counter Cluster - Mean and Std Dev Test Accuracy:",
    np.mean(bd_counter_test_accuracies),
    np.std(bd_counter_test_accuracies),
)
print(
    "BD Counter Cluster - Mean and Std Dev Balanced Test Accuracy:",
    np.mean(bd_counter_balanced_test_accuracies),
    np.std(bd_counter_balanced_test_accuracies),
)

Training with random seed 4...
Training with random seed 5...
Training with random seed 21...
Training with random seed 42...
Training with random seed 67...
Training with random seed 12...
Training with random seed 19...
Training with random seed 16...
Training with random seed 27...
Training with random seed 31...
Training with random seed 37...
Training with random seed 45...
Training with random seed 52...
Training with random seed 88...
Training with random seed 99...
BD Counter Cluster - Test Accuracies: [0.5911111111111111, 0.5022222222222222, 0.5288888888888889, 0.5555555555555556, 0.5333333333333333, 0.56, 0.5155555555555555, 0.5155555555555555, 0.5688888888888889, 0.5555555555555556, 0.49777777777777776, 0.4888888888888889, 0.5511111111111111, 0.5733333333333334, 0.5377777777777778]
BD Counter Cluster - Balanced Test Accuracies: [0.5263157894736842, 0.39285714285714285, 0.4523809523809524, 0.375, 0.4533333333333333, 0.49333333333333335, 0.5222222222222223, 0.4166666666666667,